# Build the Chatbot Android app (APK) in Google Colab

This notebook builds `Chatbot-android.apk` from the
[Robokks/ai-chatbot](https://github.com/Robokks/ai-chatbot) repo on Colab's free Linux machine.
It does not need GitHub Actions.

**How to use:** click **Runtime → Run all**. The first run takes about **15–20 minutes**. When it
finishes, the APK downloads to your computer and is also saved to your Google Drive.

> Only the Android app can be built here. A Windows `.exe` must be built on Windows.


## 1. Settings
If the repo is **private**, paste a GitHub token when asked. Create one at https://github.com/settings/tokens with *Contents: read* access to this repo. For a public repo, just press Enter.

In [ ]:
REPO = "Robokks/ai-chatbot"
BRANCH = "ccr-85a70df3-vqfuzv"
FLUTTER_VERSION = "3.47.9"
SAVE_TO_DRIVE = True   # also copy the APK to Google Drive (MyDrive/Chatbot)

from getpass import getpass
GITHUB_TOKEN = getpass("GitHub token (leave empty if the repo is public): ").strip()


## 2. Install Java 17, Flutter and the Android SDK (~8 min)

In [ ]:
import os, subprocess

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")

# Java 17 for the Android Gradle build
sh("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless unzip > /dev/null")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Flutter SDK
if not os.path.exists("/content/flutter"):
    sh(f"curl -sSfL -o /tmp/flutter.tar.xz https://storage.googleapis.com/flutter_infra_release/releases/stable/linux/flutter_linux_{FLUTTER_VERSION}-stable.tar.xz")
    sh("tar xf /tmp/flutter.tar.xz -C /content && rm /tmp/flutter.tar.xz")
    sh("git config --global --add safe.directory /content/flutter")

# Android command-line tools; Gradle downloads the exact platform/build-tools it needs
ANDROID_HOME = "/content/android-sdk"
os.environ["ANDROID_HOME"] = os.environ["ANDROID_SDK_ROOT"] = ANDROID_HOME
if not os.path.exists(f"{ANDROID_HOME}/cmdline-tools/latest"):
    sh(f"mkdir -p {ANDROID_HOME}/cmdline-tools && cd {ANDROID_HOME}/cmdline-tools && "
       "curl -sSfL -o tools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip && "
       "unzip -q tools.zip && mv cmdline-tools latest && rm tools.zip")

os.environ["PATH"] = ":".join([
    "/content/flutter/bin",
    f"{ANDROID_HOME}/cmdline-tools/latest/bin",
    f"{ANDROID_HOME}/platform-tools",
    f"{os.environ['JAVA_HOME']}/bin",
    os.environ["PATH"],
])

sh('yes | sdkmanager --licenses > /dev/null || true')
sh('sdkmanager "platform-tools" > /dev/null')
sh("flutter --disable-analytics > /dev/null 2>&1 || true")
sh(f"flutter config --android-sdk {ANDROID_HOME} --no-enable-web --no-enable-linux-desktop")
sh("yes | flutter doctor --android-licenses > /dev/null 2>&1 || true")
sh("flutter --version")


## 3. Download the app source and build the APK (~8 min)

In [ ]:
sh("rm -rf /content/ai-chatbot")
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
subprocess.run(
    f"git clone --depth 1 --branch {BRANCH} https://{auth}github.com/{REPO}.git /content/ai-chatbot",
    shell=True, check=True)
os.chdir("/content/ai-chatbot")

sh("flutter pub get")
sh("flutter test")
sh("flutter build apk --release")

APK = "/content/Chatbot-android.apk"
sh(f"cp build/app/outputs/flutter-apk/app-release.apk {APK}")
print(f"\nBuilt {APK} ({os.path.getsize(APK) / 1e6:.1f} MB)")


## 4. Download the APK

In [ ]:
import shutil
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/Chatbot", exist_ok=True)
    shutil.copy(APK, "/content/drive/MyDrive/Chatbot/Chatbot-android.apk")
    print("Saved to Google Drive: MyDrive/Chatbot/Chatbot-android.apk")

from google.colab import files
files.download(APK)


## 5. Install on your phone
1. Open **Google Drive** on your Android phone → **Chatbot** → `Chatbot-android.apk`, or copy the downloaded file to the phone.
2. Tap it, and when asked, allow **Install unknown apps** for Drive or your file manager.
3. Open **Chatbot** → menu → **Settings → API keys** and add your Claude / ChatGPT / DeepSeek key.

The APK is signed with Flutter's debug key. That's fine for installing yourself, but not for the Play Store.
